In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.classification import *
from xgboost import XGBRegressor

In [5]:
df = pd.read_csv("C:\\Users\\ripa_\\Desktop\\Programing\\IndyCar_Project\\LigaMX\\datasets\\LigaMX_dataset_v4.csv")

In [6]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Date", "Equipo"])

In [7]:
df.head()

,Date,Time,Round,Day,Venue,Opponent,Referee,Equipo,Torneo,Temporada,...,TeamAwayForm5,OpponentForm5,OpponentWinRate5,OpponentSeasonPts,OpponentHomeForm5,OpponentAwayForm5,H2HWinRate,H2HLast5,FormDiff,SeasonPtsDiff
0,2014-07-18,19:30,Jornada 1,Fri,Away,Tijuana,Luis Enrique Santander,Puebla,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
1,2014-07-18,19:30,Jornada 1,Fri,Away,Queretaro,Erick Yair Miranda,Pumas UNAM,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
2,2014-07-18,19:30,Jornada 1,Fri,Home,Pumas UNAM,Erick Yair Miranda,Queretaro,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
3,2014-07-18,19:30,Jornada 1,Fri,Home,Puebla,Luis Enrique Santander,Tijuana,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
4,2014-07-19,21:00,Jornada 1,Sat,Home,Tigres UANL,Erim Ramírez,Atlas,Apertura,2014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
unique_matches = df.drop_duplicates("match_key")[["match_key", "Date"]].sort_values("Date")
split = int(len(unique_matches)*0.95)

train_matches = unique_matches.iloc[:split]["match_key"]
unseen_matches = unique_matches.iloc[split:]["match_key"]

train_df = df[df["match_key"].isin(train_matches)]
unseen_df = df[df["match_key"].isin(unseen_matches)]

drop_cols = [
   "Date", "Time", "Round", "Day", "Venue", "Opponent", "Referee",
    "Equipo", "Torneo", "Temporada", "Result", "Points", "match_key"
]


data = train_df.drop(columns=drop_cols)
data_unseen = unseen_df.drop(columns=drop_cols)

print(data.corr(numeric_only=True)["ResultID"].sort_values())

OponentElo          -0.111008
OpponentAwayForm5   -0.028169
DayID               -0.024872
OpponentSeasonPts   -0.024218
OpponentForm5       -0.018548
OpponentWinRate5    -0.017854
TemporadaID         -0.014032
TorneoID            -0.011722
RoundID             -0.011422
TeamID              -0.003729
OpponentHomeForm5    0.001484
RefereeID            0.008243
TimeID               0.013985
TeamSeasonPts        0.023985
OpponentID           0.025848
TeamHomeForm5        0.027033
TeamWinRate5         0.051731
H2HWinRate           0.055571
TeamAwayForm5        0.057629
TeamForm5            0.060917
FormDiff             0.061126
SeasonPtsDiff        0.075997
VenueID              0.109430
H2HLast5             0.125631
TeamElo              0.145588
EloDiff              0.184402
ResultID             1.000000
Name: ResultID, dtype: float64


In [10]:
df = df.drop(columns=drop_cols)

In [11]:
print(df.columns.tolist())

['RoundID', 'VenueID', 'OpponentID', 'TeamID', 'RefereeID', 'TemporadaID', 'TorneoID', 'TimeID', 'DayID', 'ResultID', 'TeamElo', 'OponentElo', 'EloDiff', 'TeamForm5', 'TeamWinRate5', 'TeamSeasonPts', 'TeamHomeForm5', 'TeamAwayForm5', 'OpponentForm5', 'OpponentWinRate5', 'OpponentSeasonPts', 'OpponentHomeForm5', 'OpponentAwayForm5', 'H2HWinRate', 'H2HLast5', 'FormDiff', 'SeasonPtsDiff']


In [12]:
exp = setup(
    data=data, 
    target="ResultID", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    data_split_stratify=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,ResultID
2,Target type,Multiclass
3,Original data shape,"(7146, 27)"
4,Transformed data shape,"(7146, 27)"
5,Transformed train set shape,"(5002, 27)"
6,Transformed test set shape,"(2144, 27)"
7,Numeric features,26
8,Rows with missing values,5.4%
9,Preprocess,True


In [13]:
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.4967,0.0000,0.4967,0.4489,0.4351,0.2163,0.2354,0.0100
lda,Linear Discriminant Analysis,0.4943,0.0000,0.4943,0.4560,0.4432,0.2167,0.2331,0.0110
et,Extra Trees Classifier,0.4943,0.6644,0.4943,0.4827,0.4678,0.2191,0.2277,0.1070
rf,Random Forest Classifier,0.4938,0.6663,0.4938,0.4764,0.4672,0.2185,0.2253,0.1430
lr,Logistic Regression,0.4932,0.0000,0.4932,0.4670,0.4486,0.2123,0.2285,1.1380
gbc,Gradient Boosting Classifier,0.4907,0.0000,0.4907,0.4828,0.4682,0.2171,0.2244,0.5130
lightgbm,Light Gradient Boosting Machine,0.4905,0.6660,0.4905,0.4785,0.4755,0.2182,0.2221,0.4020
catboost,CatBoost Classifier,0.4890,0.6647,0.4890,0.4781,0.4742,0.2159,0.2201,2.0760
ada,Ada Boost Classifier,0.4811,0.0000,0.4811,0.4723,0.4650,0.2046,0.2099,0.0610
nb,Naive Bayes,0.4769,0.6556,0.4769,0.4678,0.4577,0.1984,0.2044,0.0160


RidgeClassifier(alpha=1.0, class_weight=None, copy_X=True, fit_intercept=True,
                max_iter=None, positive=False, random_state=123, solver='auto',
                tol=0.0001)

In [14]:
cat = create_model('catboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4736,0.6184,0.4736,0.4778,0.4691,0.1843,0.1870
1,0.4714,0.6492,0.4714,0.4712,0.4327,0.2058,0.2222
2,0.4405,0.6338,0.4405,0.4483,0.4439,0.1592,0.1594
3,0.5176,0.6605,0.5176,0.5119,0.5107,0.2634,0.2654
4,0.5000,0.6875,0.5000,0.4872,0.4918,0.2355,0.2365
5,0.4714,0.6619,0.4714,0.4618,0.4645,0.1906,0.1915
6,0.5176,0.6874,0.5176,0.4789,0.4933,0.2438,0.2472
7,0.5529,0.7371,0.5529,0.5328,0.5345,0.3019,0.3066
8,0.4868,0.6798,0.4868,0.4685,0.4610,0.1997,0.2060


In [15]:
cat_tune = tune_model(cat)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4802,0.6358,0.4802,0.4896,0.4729,0.1979,0.2030
1,0.4405,0.6695,0.4405,0.4209,0.3779,0.1592,0.1795
2,0.4912,0.6759,0.4912,0.4846,0.4871,0.2289,0.2293
3,0.4890,0.6718,0.4890,0.4826,0.4850,0.2238,0.2242
4,0.5088,0.7022,0.5088,0.4773,0.4847,0.2416,0.2463
5,0.5110,0.6782,0.5110,0.4908,0.4878,0.2421,0.2482
6,0.5396,0.7113,0.5396,0.5113,0.5168,0.2765,0.2814
7,0.5617,0.7507,0.5617,0.5386,0.5300,0.3086,0.3188
8,0.5242,0.6954,0.5242,0.5061,0.4791,0.2520,0.2663


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [29]:
predict_model(cat_tune);
#predict_model(cat);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.5415,0.7027,0.5415,0.5055,0.5021,0.2826,0.2941


In [31]:
newpred = predict_model(cat_tune, data=data_unseen)
#newpred = predict_model(cat, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.5725,0.7070,0.5725,0.5323,0.5333,0.2973,0.3114


In [16]:
rf = create_model('rf')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4736,0.6296,0.4736,0.4688,0.4618,0.1756,0.1797
1,0.4207,0.6223,0.4207,0.4437,0.3652,0.1294,0.1446
2,0.4736,0.6375,0.4736,0.4588,0.4643,0.1997,0.2006
3,0.4868,0.6572,0.4868,0.4788,0.4763,0.2142,0.2171
4,0.4912,0.6895,0.4912,0.4713,0.4773,0.2189,0.2209
5,0.4934,0.6655,0.4934,0.4604,0.4659,0.2142,0.2197
6,0.5441,0.6831,0.5441,0.5153,0.5211,0.2834,0.2883
7,0.5551,0.7257,0.5551,0.5233,0.5221,0.2985,0.3078
8,0.5308,0.6916,0.5308,0.5086,0.4943,0.2655,0.2768


In [17]:
rf_tune = tune_model(rf)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4758,0.6305,0.4758,0.4898,0.4519,0.1877,0.2009
1,0.4339,0.6546,0.4339,0.4253,0.3817,0.1495,0.1707
2,0.5154,0.6695,0.5154,0.5152,0.5136,0.2680,0.2689
3,0.4912,0.6585,0.4912,0.4938,0.4899,0.2295,0.2308
4,0.4780,0.6995,0.4780,0.4636,0.4677,0.2014,0.2029
5,0.5088,0.6736,0.5088,0.4890,0.4935,0.2437,0.2467
6,0.4978,0.6827,0.4978,0.4845,0.4900,0.2242,0.2248
7,0.5419,0.7247,0.5419,0.5192,0.5265,0.2882,0.2907
8,0.5110,0.6775,0.5110,0.4982,0.4984,0.2459,0.2491


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [33]:
predict_model(rf_tune);
#predict_model(rf);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.5182,0.6849,0.5182,0.4885,0.4947,0.2533,0.2583


In [35]:
#newpred = predict_model(rf_tune, data=data_unseen)
newpred = predict_model(rf, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.5616,0.6847,0.5616,0.5373,0.5295,0.2812,0.2935


In [18]:
lgbm = create_model('lightgbm')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4559,0.6085,0.4559,0.4597,0.4491,0.1508,0.1540
1,0.4295,0.6250,0.4295,0.4078,0.3889,0.1430,0.1551
2,0.4824,0.6496,0.4824,0.4862,0.4841,0.2207,0.2208
3,0.4670,0.6417,0.4670,0.4586,0.4614,0.1890,0.1896
4,0.4912,0.6749,0.4912,0.4875,0.4891,0.2267,0.2268
5,0.5176,0.6795,0.5176,0.5069,0.5046,0.2562,0.2598
6,0.5419,0.7009,0.5419,0.5226,0.5270,0.2842,0.2871
7,0.5441,0.7296,0.5441,0.5196,0.5261,0.2897,0.2932
8,0.5264,0.6889,0.5264,0.5132,0.5019,0.2621,0.2698


In [19]:
lgbm_tune = tune_model(lgbm)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4736,0.6287,0.4736,0.4608,0.4425,0.1566,0.1686
1,0.4405,0.6630,0.4405,0.4193,0.3633,0.1590,0.1809
2,0.5066,0.6617,0.5066,0.4862,0.4850,0.2435,0.2490
3,0.4956,0.6657,0.4956,0.4908,0.4782,0.2230,0.2287
4,0.5132,0.6994,0.5132,0.4798,0.4792,0.2432,0.2518
5,0.5198,0.6838,0.5198,0.5023,0.4663,0.2453,0.2622
6,0.5661,0.6987,0.5661,0.5592,0.5144,0.3042,0.3221
7,0.5749,0.7365,0.5749,0.5869,0.5287,0.3229,0.3411
8,0.5308,0.6999,0.5308,0.4991,0.4660,0.2580,0.2786


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [37]:
predict_model(lgbm_tune);
#predict_model(lgbm);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.5415,0.6953,0.5415,0.5160,0.4692,0.2736,0.2976


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [39]:
#newpred = predict_model(lgbm_tune, data=data_unseen)
newpred = predict_model(lgbm, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.5543,0.7006,0.5543,0.5354,0.5415,0.2874,0.2898


In [20]:
et = create_model('et')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4581,0.6250,0.4581,0.4591,0.4467,0.1544,0.1589
1,0.4449,0.6261,0.4449,0.4514,0.3903,0.1659,0.1922
2,0.4493,0.6353,0.4493,0.4449,0.4465,0.1676,0.1678
3,0.4890,0.6519,0.4890,0.4785,0.4762,0.2164,0.2197
4,0.5000,0.6815,0.5000,0.4849,0.4870,0.2314,0.2339
5,0.5022,0.6719,0.5022,0.4830,0.4824,0.2298,0.2344
6,0.5264,0.6927,0.5264,0.4774,0.4898,0.2498,0.2574
7,0.5617,0.7286,0.5617,0.5503,0.5386,0.3109,0.3188
8,0.5286,0.6886,0.5286,0.5158,0.4893,0.2602,0.2732


In [21]:
et_tune = tune_model(et)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4868,0.6442,0.4868,0.4851,0.4765,0.2011,0.2055
1,0.4295,0.6471,0.4295,0.4720,0.3564,0.1425,0.1665
2,0.4670,0.6486,0.4670,0.4511,0.4567,0.1890,0.1901
3,0.4824,0.6597,0.4824,0.4771,0.4248,0.1914,0.2075
4,0.5198,0.7075,0.5198,0.4258,0.4532,0.2459,0.2642
5,0.5154,0.6752,0.5154,0.5548,0.4434,0.2343,0.2566
6,0.5617,0.7002,0.5617,0.5639,0.4990,0.2943,0.3153
7,0.5749,0.7273,0.5749,0.6750,0.5123,0.3187,0.3434
8,0.5242,0.6905,0.5242,0.4930,0.4535,0.2462,0.2679


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [41]:
predict_model(et_tune);
#predict_model(et);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.5312,0.6923,0.5312,0.3937,0.4521,0.2559,0.2810


In [43]:
#newpred = predict_model(et_tune, data=data_unseen)
newpred = predict_model(et, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.5435,0.6985,0.5435,0.5093,0.5131,0.2558,0.2645


In [22]:
xgb = create_model('xgboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4427,0.6097,0.4427,0.4289,0.4327,0.1198,0.1209
1,0.4405,0.6231,0.4405,0.4261,0.4103,0.1597,0.1685
2,0.4471,0.6213,0.4471,0.4480,0.4476,0.1662,0.1662
3,0.4692,0.6375,0.4692,0.4601,0.4633,0.1923,0.1929
4,0.4537,0.6601,0.4537,0.4464,0.4496,0.1681,0.1684
5,0.4670,0.6527,0.4670,0.4519,0.4550,0.1802,0.1820
6,0.4912,0.6810,0.4912,0.4698,0.4784,0.2097,0.2110
7,0.5551,0.7093,0.5551,0.5240,0.5173,0.2963,0.3077
8,0.5330,0.6944,0.5330,0.5102,0.5055,0.2726,0.2806


In [23]:
xgb_tune = tune_model(xgb)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4978,0.6458,0.4978,0.4283,0.4276,0.1633,0.2008
1,0.4185,0.6631,0.4185,0.3016,0.3283,0.1258,0.1653
2,0.4868,0.6567,0.4868,0.3598,0.4062,0.1993,0.2304
3,0.4758,0.6665,0.4758,0.3506,0.3987,0.1782,0.2026
4,0.5220,0.6834,0.5220,0.3896,0.4421,0.2450,0.2752
5,0.5198,0.6742,0.5198,0.3856,0.4402,0.2404,0.2679
6,0.5617,0.7013,0.5617,0.4300,0.4855,0.2918,0.3182
7,0.5485,0.7389,0.5485,0.4162,0.4723,0.2742,0.2994
8,0.5154,0.6956,0.5154,0.3828,0.4381,0.2310,0.2550


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [45]:
#predict_model(xgb_tune);
predict_model(xgb);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.5098,0.6797,0.5098,0.4834,0.4890,0.2409,0.2449


In [47]:
#newpred = predict_model(xgb_tune, data=data_unseen)
newpred = predict_model(xgb, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.5435,0.6926,0.5435,0.5275,0.5338,0.2750,0.2762


In [24]:
nb = create_model('nb')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4648,0.6321,0.4648,0.4735,0.4510,0.1699,0.1770
1,0.4471,0.6323,0.4471,0.4457,0.3897,0.1692,0.1913
2,0.4912,0.6570,0.4912,0.5077,0.4919,0.2373,0.2405
3,0.4405,0.6265,0.4405,0.4539,0.4417,0.1588,0.1605
4,0.4868,0.6803,0.4868,0.4782,0.4801,0.2172,0.2185
5,0.4934,0.6727,0.4934,0.4764,0.4793,0.2200,0.2227
6,0.4802,0.6588,0.4802,0.4688,0.4735,0.1977,0.1983
7,0.5396,0.7127,0.5396,0.5082,0.5085,0.2749,0.2829
8,0.4758,0.6593,0.4758,0.4406,0.4426,0.1810,0.1876


In [25]:
nb_tune = tune_model(nb)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4626,0.5888,0.4626,0.4392,0.4429,0.1316,0.1340
1,0.4097,0.6081,0.4097,0.2744,0.3282,0.1126,0.1301
2,0.4912,0.6671,0.4912,0.3527,0.4106,0.2062,0.2296
3,0.4736,0.6285,0.4736,0.3432,0.3979,0.1745,0.1935
4,0.4758,0.6570,0.4758,0.3490,0.4026,0.1722,0.1898
5,0.5286,0.6638,0.5286,0.3889,0.4479,0.2544,0.2803
6,0.4934,0.6198,0.4934,0.3760,0.4267,0.1815,0.1966
7,0.5330,0.6893,0.5330,0.4027,0.4587,0.2495,0.2714
8,0.4736,0.6352,0.4736,0.3505,0.4028,0.1643,0.1804


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [49]:
#predict_model(nb_tune);
predict_model(nb);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Naive Bayes,0.5140,0.6736,0.5140,0.4818,0.4838,0.2427,0.2499


In [53]:
#newpred = predict_model(nb_tune, data=data_unseen)
newpred = predict_model(nb, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Naive Bayes,0.5290,0.6299,0.5290,0.5001,0.5047,0.2307,0.2371


In [55]:
blend1 = blend_models([cat_tune, lgbm])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4736,0.6245,0.4736,0.4753,0.4648,0.1800,0.1842
1,0.4515,0.6474,0.4515,0.4427,0.4058,0.1760,0.1936
2,0.4802,0.6644,0.4802,0.4812,0.4804,0.2162,0.2163
3,0.4912,0.6594,0.4912,0.4810,0.4840,0.2247,0.2257
4,0.5022,0.6920,0.5022,0.4901,0.4929,0.2370,0.2386
5,0.5154,0.6849,0.5154,0.5035,0.4979,0.2502,0.2553
6,0.5419,0.7105,0.5419,0.5194,0.5202,0.2792,0.2845
7,0.5441,0.7444,0.5441,0.5145,0.5188,0.2851,0.2910
8,0.5308,0.6990,0.5308,0.5208,0.4911,0.2634,0.2769


In [56]:
blend1_tune = tune_model(blend1)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4758,0.6313,0.4758,0.4830,0.4682,0.1895,0.1942
1,0.4537,0.6598,0.4537,0.4362,0.3991,0.1792,0.1992
2,0.4868,0.6713,0.4868,0.4829,0.4846,0.2240,0.2241
3,0.4934,0.6674,0.4934,0.4846,0.4870,0.2283,0.2292
4,0.4890,0.6997,0.4890,0.4683,0.4747,0.2152,0.2172
5,0.5286,0.6834,0.5286,0.5141,0.5085,0.2702,0.2763
6,0.5463,0.7127,0.5463,0.5202,0.5237,0.2865,0.2918
7,0.5661,0.7490,0.5661,0.5370,0.5353,0.3168,0.3258
8,0.5242,0.7004,0.5242,0.5135,0.4763,0.2508,0.2663


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [72]:
predict_model(blend1_tune);
#predict_model(blend1);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5373,0.7053,0.5373,0.5029,0.4991,0.2762,0.2872


In [74]:
newpred = predict_model(blend1_tune, data=data_unseen)
#newpred = predict_model(blend1, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5870,0.7105,0.5870,0.5543,0.5516,0.3236,0.3373


In [57]:
blend2 = blend_models([cat_tune, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4714,0.6356,0.4714,0.4759,0.4600,0.1822,0.1879
1,0.4692,0.6606,0.4692,0.5035,0.4127,0.2023,0.2314
2,0.4868,0.6630,0.4868,0.4831,0.4845,0.2244,0.2246
3,0.4868,0.6701,0.4868,0.4762,0.4791,0.2177,0.2188
4,0.5088,0.7029,0.5088,0.4823,0.4865,0.2413,0.2460
5,0.5176,0.6827,0.5176,0.4883,0.4867,0.2499,0.2580
6,0.5308,0.7103,0.5308,0.4878,0.4974,0.2579,0.2650
7,0.5837,0.7502,0.5837,0.5741,0.5479,0.3409,0.3548
8,0.5286,0.7025,0.5286,0.5288,0.4770,0.2566,0.2740


In [58]:
blend2_tune = tune_model(blend2)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4736,0.6352,0.4736,0.4761,0.4606,0.1845,0.1906
1,0.4626,0.6569,0.4626,0.4820,0.4056,0.1924,0.2201
2,0.4912,0.6602,0.4912,0.4881,0.4893,0.2312,0.2314
3,0.4868,0.6683,0.4868,0.4765,0.4786,0.2170,0.2184
4,0.5088,0.7020,0.5088,0.4863,0.4884,0.2416,0.2461
5,0.5198,0.6823,0.5198,0.4944,0.4918,0.2541,0.2616
6,0.5419,0.7091,0.5419,0.5014,0.5082,0.2749,0.2828
7,0.5793,0.7483,0.5793,0.5676,0.5442,0.3342,0.3475
8,0.5264,0.7020,0.5264,0.5135,0.4731,0.2531,0.2703


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [76]:
#predict_model(blend2_tune);
predict_model(blend2);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5322,0.7024,0.5322,0.4927,0.4840,0.2650,0.2793


In [78]:
#newpred = predict_model(blend2_tune, data=data_unseen)
newpred = predict_model(blend2, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5761,0.7109,0.5761,0.5414,0.5371,0.3016,0.3175


In [59]:
blend3 = blend_models([cat_tune, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4714,0.6360,0.4714,0.4812,0.4594,0.1784,0.1854
1,0.4449,0.6514,0.4449,0.4460,0.3849,0.1658,0.1880
2,0.4890,0.6712,0.4890,0.4967,0.4888,0.2309,0.2326
3,0.4648,0.6499,0.4648,0.4668,0.4639,0.1906,0.1914
4,0.4846,0.6971,0.4846,0.4628,0.4687,0.2083,0.2108
5,0.5066,0.6799,0.5066,0.4860,0.4888,0.2385,0.2422
6,0.5022,0.6872,0.5022,0.4770,0.4866,0.2242,0.2260
7,0.5639,0.7404,0.5639,0.5447,0.5231,0.3082,0.3223
8,0.5044,0.6829,0.5044,0.4858,0.4687,0.2234,0.2335


In [60]:
blend3_tune = tune_model(blend3)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4802,0.6374,0.4802,0.4917,0.4678,0.1938,0.2017
1,0.4339,0.6617,0.4339,0.4193,0.3701,0.1493,0.1693
2,0.5066,0.6763,0.5066,0.5050,0.5037,0.2545,0.2556
3,0.4890,0.6643,0.4890,0.4856,0.4865,0.2249,0.2253
4,0.5022,0.7028,0.5022,0.4769,0.4824,0.2327,0.2365
5,0.5220,0.6801,0.5220,0.5012,0.5014,0.2607,0.2658
6,0.5242,0.7037,0.5242,0.5006,0.5079,0.2571,0.2597
7,0.5727,0.7498,0.5727,0.5621,0.5380,0.3237,0.3368
8,0.5198,0.6932,0.5198,0.4904,0.4765,0.2462,0.2587


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [80]:
predict_model(blend3_tune);
#predict_model(blend3);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5336,0.7010,0.5336,0.4847,0.4872,0.2686,0.2815


In [82]:
newpred = predict_model(blend3_tune, data=data_unseen)
#newpred = predict_model(blend3, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5362,0.6864,0.5362,0.4919,0.4962,0.2369,0.2487


In [61]:
blend4 = blend_models([cat_tune, et, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4802,0.6369,0.4802,0.4931,0.4669,0.1927,0.2013
1,0.4493,0.6507,0.4493,0.4649,0.3880,0.1724,0.1985
2,0.4956,0.6672,0.4956,0.5034,0.4952,0.2410,0.2429
3,0.4824,0.6568,0.4824,0.4803,0.4799,0.2150,0.2157
4,0.4978,0.6993,0.4978,0.4767,0.4807,0.2270,0.2304
5,0.5198,0.6833,0.5198,0.4992,0.4985,0.2568,0.2623
6,0.5044,0.6932,0.5044,0.4713,0.4824,0.2230,0.2262
7,0.5727,0.7467,0.5727,0.5656,0.5340,0.3222,0.3369
8,0.5176,0.6914,0.5176,0.4865,0.4697,0.2412,0.2552


In [62]:
blend4_tune = tune_model(blend4)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4758,0.6380,0.4758,0.4850,0.4610,0.1859,0.1942
1,0.4537,0.6574,0.4537,0.4815,0.3969,0.1791,0.2065
2,0.5000,0.6671,0.5000,0.4958,0.4963,0.2442,0.2450
3,0.4868,0.6670,0.4868,0.4808,0.4819,0.2197,0.2207
4,0.4956,0.7031,0.4956,0.4762,0.4794,0.2236,0.2267
5,0.5198,0.6844,0.5198,0.4961,0.4919,0.2539,0.2616
6,0.5330,0.7052,0.5330,0.4950,0.5031,0.2628,0.2692
7,0.5881,0.7507,0.5881,0.5870,0.5529,0.3475,0.3624
8,0.5264,0.7006,0.5264,0.5136,0.4731,0.2532,0.2704


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [85]:
predict_model(blend4_tune);
#predict_model(blend4);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5373,0.7018,0.5373,0.5001,0.4875,0.2725,0.2880


In [86]:
newpred = predict_model(blend4_tune, data=data_unseen)
#newpred = predict_model(blend4, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5471,0.6975,0.5471,0.5027,0.5072,0.2557,0.2681


In [63]:
blend5 = blend_models([cat_tune, lgbm, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4714,0.6289,0.4714,0.4720,0.4630,0.1760,0.1797
1,0.4427,0.6492,0.4427,0.4290,0.3903,0.1627,0.1823
2,0.4802,0.6628,0.4802,0.4806,0.4803,0.2160,0.2160
3,0.4824,0.6632,0.4824,0.4692,0.4726,0.2096,0.2111
4,0.5000,0.6953,0.5000,0.4858,0.4896,0.2334,0.2351
5,0.5286,0.6874,0.5286,0.5098,0.5068,0.2699,0.2759
6,0.5308,0.7118,0.5308,0.4906,0.4992,0.2586,0.2653
7,0.5617,0.7463,0.5617,0.5318,0.5362,0.3131,0.3193
8,0.5330,0.7023,0.5330,0.5349,0.4900,0.2655,0.2808


In [64]:
blend5_tune = tune_model(blend5)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4736,0.6327,0.4736,0.4790,0.4652,0.1844,0.1891
1,0.4648,0.6544,0.4648,0.4647,0.4112,0.1957,0.2200
2,0.4802,0.6637,0.4802,0.4765,0.4780,0.2144,0.2145
3,0.4890,0.6669,0.4890,0.4771,0.4802,0.2202,0.2215
4,0.4912,0.6992,0.4912,0.4692,0.4739,0.2162,0.2192
5,0.5154,0.6869,0.5154,0.4916,0.4898,0.2481,0.2547
6,0.5330,0.7121,0.5330,0.4893,0.4997,0.2617,0.2688
7,0.5727,0.7485,0.5727,0.5460,0.5424,0.3272,0.3365
8,0.5352,0.7035,0.5352,0.5451,0.4871,0.2675,0.2849


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [88]:
#predict_model(blend5_tune);
predict_model(blend5);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5359,0.7047,0.5359,0.4947,0.4928,0.2728,0.2851


In [92]:
newpred = predict_model(blend5_tune, data=data_unseen)
#newpred = predict_model(blend5, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5761,0.7143,0.5761,0.5395,0.5383,0.3050,0.3190


In [65]:
blend6 = blend_models([rf, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4581,0.6338,0.4581,0.4541,0.4423,0.1539,0.1593
1,0.4493,0.6336,0.4493,0.4543,0.3920,0.1725,0.1966
2,0.4890,0.6421,0.4890,0.4807,0.4839,0.2257,0.2262
3,0.5110,0.6620,0.5110,0.5022,0.4971,0.2495,0.2540
4,0.4978,0.6943,0.4978,0.4787,0.4835,0.2282,0.2306
5,0.5132,0.6747,0.5132,0.4867,0.4867,0.2447,0.2512
6,0.5264,0.6946,0.5264,0.4762,0.4919,0.2517,0.2581
7,0.5749,0.7362,0.5749,0.5639,0.5421,0.3279,0.3400
8,0.5242,0.6965,0.5242,0.5107,0.4811,0.2522,0.2661


In [66]:
blend6_tune = tune_model(blend6)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4648,0.6341,0.4648,0.4621,0.4513,0.1636,0.1686
1,0.4493,0.6325,0.4493,0.4556,0.3925,0.1725,0.1942
2,0.4978,0.6416,0.4978,0.4917,0.4939,0.2399,0.2403
3,0.5088,0.6627,0.5088,0.5019,0.4968,0.2470,0.2512
4,0.5066,0.6954,0.5066,0.4879,0.4929,0.2423,0.2446
5,0.5198,0.6734,0.5198,0.4970,0.4933,0.2544,0.2616
6,0.5396,0.6925,0.5396,0.5070,0.5145,0.2758,0.2809
7,0.5617,0.7350,0.5617,0.5416,0.5296,0.3080,0.3184
8,0.5308,0.6967,0.5308,0.5210,0.4911,0.2635,0.2770


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [95]:
#predict_model(blend6_tune);
predict_model(blend6);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5303,0.6864,0.5303,0.4974,0.4860,0.2628,0.2760


In [97]:
#newpred = predict_model(blend6_tune, data=data_unseen)
newpred = predict_model(blend6, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5725,0.6997,0.5725,0.5459,0.5370,0.2973,0.3118


In [67]:
blend7 = blend_models([lgbm, xgb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4471,0.6096,0.4471,0.4365,0.4369,0.1287,0.1304
1,0.4383,0.6293,0.4383,0.4185,0.4047,0.1564,0.1661
2,0.4626,0.6357,0.4626,0.4665,0.4644,0.1907,0.1907
3,0.4868,0.6431,0.4868,0.4760,0.4795,0.2182,0.2191
4,0.4604,0.6710,0.4604,0.4543,0.4569,0.1787,0.1789
5,0.5022,0.6695,0.5022,0.4867,0.4889,0.2336,0.2362
6,0.5286,0.6944,0.5286,0.5034,0.5112,0.2636,0.2665
7,0.5595,0.7235,0.5595,0.5330,0.5309,0.3071,0.3154
8,0.5286,0.6957,0.5286,0.5110,0.4994,0.2642,0.2732


In [68]:
blend7_tune = tune_model(blend7)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4626,0.6100,0.4626,0.4588,0.4546,0.1559,0.1583
1,0.4361,0.6295,0.4361,0.4143,0.4003,0.1530,0.1637
2,0.4604,0.6408,0.4604,0.4627,0.4615,0.1867,0.1868
3,0.4912,0.6440,0.4912,0.4817,0.4845,0.2249,0.2259
4,0.4714,0.6731,0.4714,0.4668,0.4687,0.1960,0.1962
5,0.4956,0.6730,0.4956,0.4845,0.4838,0.2231,0.2258
6,0.5286,0.6974,0.5286,0.5048,0.5106,0.2621,0.2655
7,0.5683,0.7267,0.5683,0.5420,0.5423,0.3225,0.3298
8,0.5286,0.6945,0.5286,0.5136,0.5007,0.2644,0.2733


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [99]:
predict_model(blend7_tune);
#predict_model(blend7);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5266,0.6932,0.5266,0.4946,0.4985,0.2637,0.2703


In [101]:
#newpred = predict_model(blend7_tune, data=data_unseen)
newpred = predict_model(blend7, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5616,0.7038,0.5616,0.5414,0.5479,0.2999,0.3025


In [69]:
blend8 = blend_models([cat_tune, rf, lgbm, et, xgb, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4670,0.6303,0.4670,0.4580,0.4533,0.1607,0.1646
1,0.4471,0.6482,0.4471,0.4443,0.3918,0.1692,0.1913
2,0.4758,0.6562,0.4758,0.4731,0.4738,0.2079,0.2082
3,0.5066,0.6602,0.5066,0.4959,0.4975,0.2466,0.2485
4,0.4802,0.6947,0.4802,0.4620,0.4681,0.2030,0.2045
5,0.5132,0.6824,0.5132,0.4913,0.4893,0.2452,0.2514
6,0.5308,0.7011,0.5308,0.4942,0.5056,0.2630,0.2674
7,0.5749,0.7440,0.5749,0.5639,0.5421,0.3279,0.3400
8,0.5264,0.7036,0.5264,0.5158,0.4806,0.2548,0.2699


In [70]:
blend8_tune = tune_model(blend8)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4714,0.6307,0.4714,0.4622,0.4578,0.1676,0.1715
1,0.4493,0.6483,0.4493,0.4416,0.3949,0.1725,0.1935
2,0.4736,0.6534,0.4736,0.4687,0.4704,0.2036,0.2039
3,0.5022,0.6605,0.5022,0.4910,0.4931,0.2401,0.2418
4,0.4868,0.6937,0.4868,0.4722,0.4774,0.2148,0.2158
5,0.5154,0.6797,0.5154,0.4924,0.4916,0.2489,0.2549
6,0.5286,0.6996,0.5286,0.4923,0.5040,0.2599,0.2641
7,0.5793,0.7424,0.5793,0.5702,0.5459,0.3346,0.3473
8,0.5308,0.7040,0.5308,0.5198,0.4866,0.2623,0.2771


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [103]:
#predict_model(blend8_tune);
predict_model(blend8);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5368,0.7014,0.5368,0.4941,0.4909,0.2734,0.2867


In [105]:
newpred = predict_model(blend8_tune, data=data_unseen)
#newpred = predict_model(blend8, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5616,0.6982,0.5616,0.5349,0.5355,0.2834,0.2925


In [106]:
blend9 = blend_models([cat_tune, xgb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4537,0.6246,0.4537,0.4487,0.4472,0.1437,0.1452
1,0.4493,0.6492,0.4493,0.4336,0.4074,0.1727,0.1865
2,0.4537,0.6428,0.4537,0.4533,0.4535,0.1755,0.1755
3,0.4868,0.6574,0.4868,0.4757,0.4797,0.2186,0.2194
4,0.4670,0.6820,0.4670,0.4502,0.4557,0.1828,0.1840
5,0.4912,0.6668,0.4912,0.4686,0.4725,0.2144,0.2179
6,0.5198,0.6974,0.5198,0.4899,0.4998,0.2487,0.2518
7,0.5661,0.7351,0.5661,0.5422,0.5279,0.3130,0.3257
8,0.5396,0.7026,0.5396,0.5203,0.5016,0.2786,0.2913


In [107]:
blend9_tune = tune_model(blend9)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4846,0.6316,0.4846,0.4846,0.4784,0.1959,0.1988
1,0.4581,0.6628,0.4581,0.4450,0.4017,0.1858,0.2069
2,0.4736,0.6590,0.4736,0.4703,0.4717,0.2042,0.2043
3,0.5022,0.6681,0.5022,0.4886,0.4920,0.2399,0.2416
4,0.5044,0.6953,0.5044,0.4835,0.4885,0.2376,0.2404
5,0.5132,0.6746,0.5132,0.4903,0.4932,0.2479,0.2523
6,0.5308,0.7065,0.5308,0.4972,0.5051,0.2616,0.2667
7,0.5727,0.7470,0.5727,0.5465,0.5360,0.3244,0.3364
8,0.5419,0.7025,0.5419,0.5237,0.4944,0.2793,0.2955


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [110]:
predict_model(blend9_tune);
#predict_model(blend9);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5326,0.7039,0.5326,0.4888,0.4920,0.2689,0.2796


In [111]:
newpred = predict_model(blend9_tune, data=data_unseen)
#newpred = predict_model(blend9, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5725,0.7126,0.5725,0.5467,0.5487,0.3045,0.3125


In [112]:
blend10 = blend_models([cat_tune, lgbm, xgb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4581,0.6195,0.4581,0.4504,0.4489,0.1484,0.1505
1,0.4537,0.6430,0.4537,0.4416,0.4151,0.1794,0.1940
2,0.4714,0.6481,0.4714,0.4723,0.4718,0.2027,0.2027
3,0.4824,0.6544,0.4824,0.4691,0.4733,0.2106,0.2117
4,0.4670,0.6821,0.4670,0.4591,0.4622,0.1874,0.1878
5,0.5044,0.6746,0.5044,0.4850,0.4878,0.2356,0.2390
6,0.5352,0.7017,0.5352,0.5052,0.5124,0.2699,0.2745
7,0.5683,0.7366,0.5683,0.5425,0.5364,0.3194,0.3294
8,0.5220,0.7016,0.5220,0.5059,0.4859,0.2510,0.2623


In [113]:
blend10_tune = tune_model(blend10)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4780,0.6260,0.4780,0.4812,0.4716,0.1874,0.1908
1,0.4405,0.6520,0.4405,0.4252,0.3922,0.1594,0.1759
2,0.4890,0.6614,0.4890,0.4879,0.4884,0.2285,0.2285
3,0.4956,0.6617,0.4956,0.4841,0.4869,0.2303,0.2317
4,0.4824,0.6920,0.4824,0.4676,0.4722,0.2069,0.2082
5,0.5088,0.6817,0.5088,0.4902,0.4895,0.2403,0.2449
6,0.5352,0.7091,0.5352,0.5100,0.5137,0.2696,0.2743
7,0.5595,0.7451,0.5595,0.5301,0.5295,0.3067,0.3151
8,0.5286,0.7011,0.5286,0.5226,0.4888,0.2597,0.2733


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [117]:
predict_model(blend10_tune);
#predict_model(blend10);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5350,0.7043,0.5350,0.4990,0.4975,0.2730,0.2834


In [119]:
newpred = predict_model(blend10_tune, data=data_unseen)
newpred = predict_model(blend10, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5870,0.7127,0.5870,0.5646,0.5672,0.3303,0.3370


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5616,0.7110,0.5616,0.5386,0.5459,0.2974,0.3006


In [120]:
save_model(blend10_tune, "LigaMX_model_v3")

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['RoundID', 'VenueID', 'OpponentID',
                                              'TeamID', 'RefereeID',
                                              'TemporadaID', 'TorneoID',
                                              'TimeID', 'DayID', 'TeamElo',
                                              'OponentElo', 'EloDiff',
                                              'TeamForm5', 'TeamWinRate5',
                                              'TeamSeasonPts', 'TeamHomeForm5',
                                              'TeamAwayForm5', 'OpponentForm5',
                                              'OpponentWinRate5',
                                              'Opp...
                                                              max_cat_threshold=None,
                                                          